In [18]:
import inspect
import itertools
import numpy as np
import random as rd
import pandas as pd
import pandas_ta as ta
from backtesting import Backtest, Strategy

In [19]:
def analyze_function(func):
    """
    Analyzes the function to extract parameter names and default values.

    Args:
    - func (callable): The function to analyze.

    Returns:
    - dict: A dictionary with parameter names and their default values.
    """
    signature = inspect.signature(func)
    parameters = {
        param.name: param.default
        for param in signature.parameters.values()
        if param.default is not param.empty
    }
    return parameters
    
def generate_random_parameters(base_values={"length": 1}, min_value=1, max_value=200):
    """
    Generates the maximum number of unique parameter sets, ensuring each generated value 
    is greater than its base value.

    Args:
    - base_values (dict): A dictionary with base parameter names and their base values.
    - min_value (int): The minimum value for the parameters.
    - max_value (int): The maximum value for the parameters.

    Returns:
    - List[dict]: A list of dictionaries containing the parameters.
    """
    if len(base_values) == 1:
        # Only one parameter
        key, base_value = next(iter(base_values.items()))
        effective_min_value = max(base_value + 1, min_value)
        possible_values = list(range(effective_min_value, max_value + 1))
        
        if not possible_values:
            raise ValueError("No possible values to generate.")
        
        return [{key: value} for value in possible_values]
    else:
        # Multiple parameters
        possible_values_dict = {
            key: list(range(max(base_value + 1, min_value), max_value + 1))
            for key, base_value in base_values.items()
        }
        
        # Compute the Cartesian product of all possible values
        all_keys = list(possible_values_dict.keys())
        all_combinations = itertools.product(*possible_values_dict.values())
        
        return [dict(zip(all_keys, combination)) for combination in all_combinations]


def extract_parameters(parameters):
    extracted_params = {}

    for indicator in parameters:
        for key, value in indicator.items():
            extracted_params[key] = []
            for param_set in value:
                extracted_params[key].append(param_set)

    return extracted_params

def extract_list_to_key_value(pairs):
    for key, value in pairs.items():
        return key, value
    return pairs


In [29]:
    # for indicator_name in momentum_indicators:
    #     ind = momentum.get_indicator(indicator_name)
    #     # print(f'is callable: {callable(ind)}')
    #     # Analyze the function to get base parameter values
    #     base_values = analyze_function(ind)
    #     # print(f'{idx}: base_values: {base_values}')

new_parameters = []
# Generate 200 day moving avarage sets of random parameters
parameters = generate_random_parameters(base_values={'length': 1}, min_value=1, max_value=200)

# print(f'{idx}: parameters: {parameters}')

new_parameters.append({"rsi": parameters})


new_parameters

[{'rsi': [{'length': 2},
   {'length': 3},
   {'length': 4},
   {'length': 5},
   {'length': 6},
   {'length': 7},
   {'length': 8},
   {'length': 9},
   {'length': 10},
   {'length': 11},
   {'length': 12},
   {'length': 13},
   {'length': 14},
   {'length': 15},
   {'length': 16},
   {'length': 17},
   {'length': 18},
   {'length': 19},
   {'length': 20},
   {'length': 21},
   {'length': 22},
   {'length': 23},
   {'length': 24},
   {'length': 25},
   {'length': 26},
   {'length': 27},
   {'length': 28},
   {'length': 29},
   {'length': 30},
   {'length': 31},
   {'length': 32},
   {'length': 33},
   {'length': 34},
   {'length': 35},
   {'length': 36},
   {'length': 37},
   {'length': 38},
   {'length': 39},
   {'length': 40},
   {'length': 41},
   {'length': 42},
   {'length': 43},
   {'length': 44},
   {'length': 45},
   {'length': 46},
   {'length': 47},
   {'length': 48},
   {'length': 49},
   {'length': 50},
   {'length': 51},
   {'length': 52},
   {'length': 53},
   {'length':

In [45]:
def standardize_columns(df):
    """
    Ensures that the DataFrame has columns named 'Open', 'High', 'Low', 'Close', and 'Volume'.
    If the columns are lowercase or differently named, they will be renamed accordingly.
    """
    # Define expected column mappings for both lowercase and uppercase
    expected_columns = {
        'open': 'Open',
        'high': 'High',
        'low': 'Low',
        'close': 'Close',
        'volume': 'Volume',
        'adj close': "Adj Close"
    }
    
    # Create a mapping for renaming based on the existing columns in df
    rename_columns = {col: expected_columns[col.lower()] for col in df.columns if col.lower() in expected_columns}
    
    # Rename the columns to match the expected format
    df = df.rename(columns=rename_columns)
    
    # Verify if required columns are present after renaming
    missing_columns = [col for col in ['Open', 'High', 'Low', 'Close', 'Adj Close'] if col not in df.columns]
    if missing_columns:
        raise ValueError(f"Data is missing required columns: {missing_columns}")

    return df

In [55]:
# Load data
data = pd.read_csv("./data/usa/processed.csv")
df = standardize_columns(data)
df

,datetime,Adj Close,Close,High,Low,Open,Volume,ticker
0,2019-10-28 00:00:00+00:00,110.700897,118.580002,119.300003,118.070000,118.639999,2552700,AXP
1,2019-10-28 00:00:00+00:00,192.536774,217.750000,218.919998,215.690002,216.320007,2707800,GS
2,2019-10-28 00:00:00+00:00,155.702164,172.679993,173.720001,171.970001,171.970001,2367200,HON
3,2019-10-28 00:00:00+00:00,45.673771,60.180000,60.660000,60.040001,60.450001,14621400,VZ
4,2019-10-28 00:00:00+00:00,36.785854,39.740002,39.900002,39.509998,39.750000,10213800,WMT
...,...,...,...,...,...,...,...,...
46645,2024-10-25 19:30:00+00:00,41.380001,41.380001,41.500000,41.369999,41.493999,3783338,VZ
46646,2024-10-25 19:30:00+00:00,155.029999,155.029999,155.210007,154.830994,155.190002,826646,BA
46647,2024-10-25 19:30:00+00:00,103.970001,103.970001,104.262802,103.860001,104.254997,1469329,MRK
46648,2024-10-25 19:30:00+00:00,207.970993,207.970993,208.100006,207.619995,208.070007,651098,HON


In [54]:

# Indicator parameters, e.g., length for RSI
indicator_params = {'length': 14}
parameter = 14  # Central parameter for generating thresholds


# for params in 

indicator_series = ta.rsi(close=df["Volume"], length=1)

# df
indicator_series

0          NaN
1        100.0
2          0.0
3        100.0
4          0.0
         ...  
46645    100.0
46646      0.0
46647    100.0
46648      0.0
46649    100.0
Name: RSI_1, Length: 46650, dtype: float64

In [50]:
class BStrategy(Strategy):
    # Define class attributes to be set dynamically

    # indicator_params = None
    # check_field = None
    threshold = None
    indicator_series = None
    
    def init(self):
        """
        Initialize the indicator dynamically using the provided pandas-ta function.
        """
        # Check if indicator_series is valid
        if self.indicator_series is not None and len(self.indicator_series) == len(self.data.df):
            # Convert to numpy array and assign to self.indicator
            self.indicator = self.I(lambda: self.indicator_series.to_numpy())
        else:
            # If the length doesn't match or computation failed, raise an error with details
            raise ValueError("Indicator computation failed or resulted in incorrect length.")
        
        print(f"threshold: {self.threshold}")
    
    def next(self):
        """
        Execute buy/sell signals based on provided conditions.
        """
        # Check buy condition
        if self.indicator[-1] > self.threshold:
            self.buy()
                
        if self.indicator[-1] < self.threshold:
            self.sell()


class Backtesting:
    def __init__(self, data, cash=10000, commission=0.001):
        """
        Initialize the backtesting framework.

        Parameters:
        - data: The historical data as a DataFrame.
        - cash: Initial cash balance for backtesting.
        - commission: Commission rate per trade.
        """
        self.data = data
        self.cash = cash
        self.commission = commission
    
    def generate_random_conditions(self, parameter):
        """
        Generate random buy threshold around a parameter for the indicator.

        Parameters:
        - parameter: The central parameter value to determine thresholds.

        Returns:
        - threshold: A randomly chosen threshold for buy condition.
        """
        # Define range around the parameter for thresholds
        threshold = rd.randint(parameter - 5, parameter + 5)

        return threshold

    def run_backtest(self, indicator_series, parameter):
        """
        Run the backtest with a single indicator and random buy/sell conditions around a parameter.

        Parameters:
        - indicator_func: A pandas-ta function like ta.rsi or ta.sma.
        - indicator_params: Parameters for the indicator function (e.g., {'length': 14}).
        - parameter: Central value for threshold randomization.
        - check_field: The column in `data` on which the indicator operates (e.g., 'Close').
        - indicator_name: Name of the indicator for labeling (optional).

        Returns:
        - stats: Performance statistics of the backtest.
        """
        # Generate random buy threshold
        threshold = self.generate_random_conditions(parameter)

        print(threshold)
        
        # Initialize the Backtest instance with parameters for DynamicStrategy
        bt = Backtest(self.data, BStrategy, cash=self.cash, commission=self.commission)        
        # Run the backtest and pass parameters
        stats = bt.run(
            indicator_series=indicator_series,
            threshold=threshold
        )

        return stats


# Initialize and run the dynamic backtest with random conditions around a parameter
dynamic_backtest = Backtesting(df)
stats = dynamic_backtest.run_backtest(indicator_series, parameter)

# Print the performance statistics
print(stats)

13
threshold: 13
Start                                     0.0
End                                   46649.0
Duration                              46649.0
Exposure Time [%]                    0.017149
Equity Final [$]                          0.0
Equity Peak [$]                  50157.839133
Return [%]                             -100.0
Buy & Hold Return [%]              -52.985327
Return (Ann.) [%]                         0.0
Volatility (Ann.) [%]                     NaN
Sharpe Ratio                              NaN
Sortino Ratio                             NaN
Calmar Ratio                              0.0
Max. Drawdown [%]                      -100.0
Avg. Drawdown [%]                  -50.739711
Max. Drawdown Duration                46640.0
Avg. Drawdown Duration                23321.0
# Trades                                  6.0
Win Rate [%]                        83.333333
Best Trade [%]                     279.111847
Worst Trade [%]                   -120.689613
Avg. Trade [%]   

/var/folders/4q/r63zv7yj5gz55km0lrg80gwc0000gn/T/ipykernel_9046/3651092169.py:84: UserWarning: Data index is not datetime. Assuming simple periods, but `pd.DateTimeIndex` is advised.
  bt = Backtest(self.data, BStrategy, cash=self.cash, commission=self.commission)


In [42]:
def call_indicators(ind_name="rsi", base_values={"length": 1}, min_value = 1, max_value = 200):
    # usage
    new_parameters = []
    # base_values = analyze_function(ind_func)
    # Generate and append random parameters
    parameters = generate_random_parameters(base_values=base_values, min_value=min_value, max_value=max_value)
    new_parameters.append({ind_name: parameters})
    
    # Extract parameters
    params = extract_parameters(new_parameters)
    
    # Print extracted parameters
    for name, parameters in params.items():
        print(name)
        for idx, parameter in enumerate(list(parameters)):
            # Replace with actual function call if needed
            print(f"{idx}: {parameter["length"]}")

            rsi = ta.rsi(close=df["Close"], length=parameter["length"])
            
            print(rsi)

call_indicators("rsi")

rsi
0: 2
0              NaN
1              NaN
2        52.384973
3        15.512037
4        12.352553
           ...    
46645    31.234965
46646    45.189814
46647    38.220430
46648    62.057958
46649    29.142133
Name: RSI_2, Length: 46650, dtype: float64
1: 3
0              NaN
1              NaN
2              NaN
3        23.617528
4        20.284934
           ...    
46645    38.162858
46646    46.044974
46647    42.402569
46648    53.613581
46649    37.561976
Name: RSI_3, Length: 46650, dtype: float64
2: 4
0              NaN
1              NaN
2              NaN
3              NaN
4        24.323441
           ...    
46645    41.236760
46646    46.972327
46647    44.377640
46648    51.633414
46649    41.156208
Name: RSI_4, Length: 46650, dtype: float64
3: 5
0              NaN
1              NaN
2              NaN
3              NaN
4              NaN
           ...    
46645    42.955426
46646    47.570947
46647    45.503327
46648    50.933131
46649    43.080090
Name: RSI_5

In [ ]:
def ris_cal()

In [42]:
# It's also an ok backtester


# import numpy as np
# import random as rd
# import pandas as pd
# import pandas_ta as ta
# from backtesting import Backtest, Strategy
# from backtesting.test import GOOG


# class DynamicStrategy(Strategy):
#     # Define class attributes to be set dynamically
#     indicator_func = None
#     indicator_params = None
#     check_field = None
#     threshold = None

#     def init(self):
#         """
#         Initialize the indicator dynamically using the provided pandas-ta function.
#         """
#         # Ensure that indicator_func, indicator_params, and check_field are set
#         assert self.indicator_func is not None, "indicator_func is not set"
#         assert self.indicator_params is not None, "indicator_params is not set"
#         assert self.check_field is not None, "check_field is not set"

#         # Access the underlying DataFrame from self.data
#         data_series = self.data.df[self.check_field]

#         # Debug: Print the first few values of the input data
#         # print("Input data for indicator computation:")
#         # print(data_series.head())

#         # Compute the indicator
#         indicator_series = self.indicator_func(data_series, **self.indicator_params)
        
#         # Debug: Check the output of the indicator
#         # print("Computed indicator values (first 5 rows):")
#         # print(indicator_series.head() if indicator_series is not None else "Indicator computation returned None")

#         # Check if indicator_series is valid
#         if indicator_series is not None and len(indicator_series) == len(self.data.df):
#             # Convert to numpy array and assign to self.indicator
#             self.indicator = self.I(lambda: indicator_series.to_numpy())
#         else:
#             # If the length doesn't match or computation failed, raise an error with details
#             raise ValueError("Indicator computation failed or resulted in incorrect length.")
        
#         print(f"threshold: {self.threshold}")
    
#     def next(self):
#         """
#         Execute buy/sell signals based on provided conditions.
#         """
#         # Check buy condition
#         if self.indicator[-1] > self.threshold:
#             self.buy()
                
#         if self.indicator[-1] < self.threshold:
#             self.sell()


# class DynamicBacktest:
#     def __init__(self, data, cash=10000, commission=0.001):
#         """
#         Initialize the backtesting framework.

#         Parameters:
#         - data: The historical data as a DataFrame.
#         - cash: Initial cash balance for backtesting.
#         - commission: Commission rate per trade.
#         """
#         self.data = data
#         self.cash = cash
#         self.commission = commission
    
#     def generate_random_conditions(self, parameter):
#         """
#         Generate random buy threshold around a parameter for the indicator.

#         Parameters:
#         - parameter: The central parameter value to determine thresholds.

#         Returns:
#         - threshold: A randomly chosen threshold for buy condition.
#         """
#         # Define range around the parameter for thresholds
#         threshold = rd.randint(parameter - 5, parameter + 5)
#         return threshold

#     def run_backtest(self, indicator_func, indicator_params, parameter, check_field, indicator_name="None"):
#         """
#         Run the backtest with a single indicator and random buy/sell conditions around a parameter.

#         Parameters:
#         - indicator_func: A pandas-ta function like ta.rsi or ta.sma.
#         - indicator_params: Parameters for the indicator function (e.g., {'length': 14}).
#         - parameter: Central value for threshold randomization.
#         - check_field: The column in `data` on which the indicator operates (e.g., 'Close').
#         - indicator_name: Name of the indicator for labeling (optional).

#         Returns:
#         - stats: Performance statistics of the backtest.
#         """
#         # Generate random buy threshold
#         threshold = self.generate_random_conditions(parameter)

        
        
#         # Initialize the Backtest instance with parameters for DynamicStrategy
#         bt = Backtest(self.data, DynamicStrategy, cash=self.cash, commission=self.commission)        
#         # Run the backtest and pass parameters
#         stats = bt.run(
#             indicator_func=indicator_func,
#             indicator_params=indicator_params,
#             check_field=check_field,
#             threshold=threshold
#         )

#         return stats


# def standardize_columns(df):
#     """
#     Ensures that the DataFrame has columns named 'Open', 'High', 'Low', 'Close', and 'Volume'.
#     If the columns are lowercase or differently named, they will be renamed accordingly.
#     """
#     # Define expected column mappings for both lowercase and uppercase
#     expected_columns = {
#         'open': 'Open',
#         'high': 'High',
#         'low': 'Low',
#         'close': 'Close',
#         'volume': 'Volume'
#     }
    
#     # Create a mapping for renaming based on the existing columns in df
#     rename_columns = {col: expected_columns[col.lower()] for col in df.columns if col.lower() in expected_columns}
    
#     # Rename the columns to match the expected format
#     df = df.rename(columns=rename_columns)
    
#     # Verify if required columns are present after renaming
#     missing_columns = [col for col in ['Open', 'High', 'Low', 'Close'] if col not in df.columns]
#     if missing_columns:
#         raise ValueError(f"Data is missing required columns: {missing_columns}")

#     return df

# # Load data
# data = pd.read_csv("./data/usa/processed.csv")
# df = standardize_columns(data)

# # Indicator parameters, e.g., length for RSI
# indicator_params = {'length': 14}
# parameter = 14  # Central parameter for generating thresholds

# # Initialize and run the dynamic backtest with random conditions around a parameter
# dynamic_backtest = DynamicBacktest(df)
# stats = dynamic_backtest.run_backtest(ta.rsi, indicator_params, parameter, "Close", indicator_name="RSI")

# # Print the performance statistics
# print(stats)



In [11]:
# import numpy as np
# import random as rd
# import pandas as pd
# import pandas_ta as ta
# from backtesting import Backtest, Strategy
# from backtesting.test import GOOG


# class DynamicStrategy(Strategy):
#     # Define the attributes with default values
#     indicator_func = None
#     indicator_params = {}
#     check_field = "Close"
#     threshold = None

#     def __init__(self, *args, **kwargs):
#         # Ensure that any dynamically passed parameters are correctly set
#         super().__init__(*args, **kwargs)
#         self.indicator_func = kwargs.get("indicator_func", self.indicator_func)
#         self.indicator_params = kwargs.get("indicator_params", self.indicator_params)
#         self.check_field = kwargs.get("check_field", self.check_field)
#         self.threshold = kwargs.get("threshold", self.threshold)

#     def init(self):
#         # Ensure that check_field and indicator_func are set
#         assert self.check_field is not None, "check_field is not set"
#         assert self.indicator_func is not None, "indicator_func is not set"

#         # Access the underlying DataFrame from self.data
#         data_series = self.data.df[self.check_field]

#         # Compute the indicator
#         indicator_series = self.indicator_func(data_series, **self.indicator_params)

#         # Check if indicator_series is valid
#         if indicator_series is not None and len(indicator_series) == len(self.data.df):
#             # Convert to numpy array and assign to self.indicator
#             self.indicator = self.I(lambda: indicator_series.to_numpy())
#         else:
#             raise ValueError("Indicator computation failed or resulted in incorrect length.")
    
#     def next(self):
#         """
#         Execute buy/sell signals based on provided conditions.
#         """
#         # Check buy/sell conditions based on threshold
#         if self.indicator[-1] > self.threshold:
#             self.buy()
#         elif self.indicator[-1] < self.threshold:
#             self.sell()


# class DynamicBacktest:
#     def __init__(self, data, cash=10000, commission=0.001):
#         # Ensure index is datetime to prevent warnings
#         self.data = data
#         if not pd.api.types.is_datetime64_any_dtype(self.data.index):
#             self.data.index = pd.to_datetime(self.data.index, errors='coerce')
        
#         self.cash = cash
#         self.commission = commission
    
#     def generate_random_conditions(self, parameter):
#         """
#         Generate a random threshold around the given parameter.
#         """
#         return rd.randint(parameter - 5, parameter + 5)

#     def run_backtest(self, indicator_func, indicator_params, parameter, check_field="Close", optimize=False):
#         """
#         Run the backtest with a single indicator and optional parameter optimization.
#         """
#         # Set initial threshold
#         threshold = self.generate_random_conditions(parameter)

#         # Initialize the Backtest instance with parameters for DynamicStrategy
#         bt = Backtest(self.data, DynamicStrategy, cash=self.cash, commission=self.commission)

#         if optimize:
#             # Optimize threshold parameter
#             stats = bt.optimize(
#                 threshold=range(threshold - 10, threshold + 10, 1),
#                 indicator_func=indicator_func,
#                 indicator_params=indicator_params,
#                 check_field=check_field,
#                 maximize='Sharpe Ratio'  # Example metric for optimization
#             )
#         else:
#             # Run the backtest with fixed parameters
#             stats = bt.run(
#                 indicator_func=indicator_func,
#                 indicator_params=indicator_params,
#                 check_field=check_field,
#                 threshold=threshold
#             )

#         return stats


# def standardize_columns(df):
#     """
#     Ensures that the DataFrame has columns named 'Open', 'High', 'Low', 'Close', and 'Volume'.
#     If the columns are lowercase or differently named, they will be renamed accordingly.
#     """
#     expected_columns = {
#         'open': 'Open',
#         'high': 'High',
#         'low': 'Low',
#         'close': 'Close',
#         'volume': 'Volume'
#     }
    
#     rename_columns = {col: expected_columns[col.lower()] for col in df.columns if col.lower() in expected_columns}
#     df = df.rename(columns=rename_columns)
    
#     missing_columns = [col for col in ['Open', 'High', 'Low', 'Close'] if col not in df.columns]
#     if missing_columns:
#         raise ValueError(f"Data is missing required columns: {missing_columns}")

#     return df


# # Load and standardize data
# data = pd.read_csv("./data/usa/processed.csv")
# df = standardize_columns(data)

# # Define parameters
# indicator_params = {'length': 14}
# parameter = 14  # Central parameter for generating thresholds

# # Initialize and run the dynamic backtest with optimization
# dynamic_backtest = DynamicBacktest(df)
# stats = dynamic_backtest.run_backtest(
#     indicator_func=ta.rsi,
#     indicator_params=indicator_params,
#     parameter=parameter,
#     check_field="Close",
#     optimize=True  # Set to True to perform optimization
# )

# # Print the optimized performance statistics
# print(stats)
